In [1]:
!pip install -q torch torchvision tqdm



In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import os


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [7]:
!unzip -q classification_data.zip

In [18]:
DATA_DIR = "/content/classification_data"
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 15


In [19]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomAffine(0, shear=10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [21]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Classes:", train_dataset.classes)
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

Classes: ['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'truck']
Train samples: 7451
Val samples: 1299


In [22]:
model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.classifier[1].in_features,
              len(train_dataset.classes))
)

model = model.to(device)

In [23]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_350/2487718576.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [24]:
def train_epoch():
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(train_loader), correct / total

In [25]:
def validate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [26]:
best_val_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch()
    val_acc = validate()

    scheduler.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {train_loss:.4f} "
          f"Train Acc: {train_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "efficientnetb0_best.pth")
        print("Saved new best model")

print("Best Validation Accuracy:", best_val_acc)

/tmp/ipykernel_350/1748641115.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1/15] Loss: 2.5835 Train Acc: 0.2975 Val Acc: 0.4935
Saved new best model
Epoch [2/15] Loss: 2.0751 Train Acc: 0.4640 Val Acc: 0.5574
Saved new best model
Epoch [3/15] Loss: 1.8654 Train Acc: 0.5393 Val Acc: 0.5751
Saved new best model
Epoch [4/15] Loss: 1.7317 Train Acc: 0.5881 Val Acc: 0.5897
Saved new best model
Epoch [5/15] Loss: 1.6058 Train Acc: 0.6391 Val Acc: 0.6066
Saved new best model
Epoch [6/15] Loss: 1.4898 Train Acc: 0.6834 Val Acc: 0.6259
Saved new best model
Epoch [7/15] Loss: 1.4037 Train Acc: 0.7155 Val Acc: 0.6274
Saved new best model
Epoch [8/15] Loss: 1.3067 Train Acc: 0.7588 Val Acc: 0.6305
Saved new best model
Epoch [9/15] Loss: 1.2340 Train Acc: 0.7920 Val Acc: 0.6259
Epoch [10/15] Loss: 1.1875 Train Acc: 0.8126 Val Acc: 0.6397
Saved new best model
Epoch [11/15] Loss: 1.1518 Train Acc: 0.8251 Val Acc: 0.6428
Saved new best model
Epoch [12/15] Loss: 1.1018 Train Acc: 0.8455 Val Acc: 0.6397
Epoch [13/15] Loss: 1.0851 Train Acc: 0.8508 Val Acc: 0.6467
Saved 

In [27]:
from google.colab import files
files.download("efficientnetb0_best.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>